<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_14/dataset_pbl/praktikum_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 1**

## **Langkah 1 - Import Library**

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## **Langkah 2 - Pra Pengolahan Data**
### **Langkah 2.1 - Pra Pengolahan Data Training**

In [2]:
# Load dataset Sleep Health
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')

fitur = [
    'Age', 'Sleep Duration', 'Quality of Sleep',
    'Physical Activity Level', 'Stress Level',
    'Heart Rate', 'Daily Steps'
]
X = df[fitur].values
y = df['Sleep Disorder'].apply(lambda x: 0 if pd.isna(x) or x == 'None' else 1).values

# Split dataset menjadi training dan testing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalisasi dan reshape khusus data training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

print("Shape X_train:", X_train.shape)

Shape X_train: (299, 7, 1)


### **Langkah 2.2 - Pra Pengolahan Data Testing**

In [3]:
# Normalisasi (menggunakan scaler dari data training) dan reshape khusus data testing
X_test = scaler.transform(X_test_raw)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("Shape X_test:", X_test.shape)

Shape X_test: (75, 7, 1)


## **Langkah 3 - Pembuatan Model CNN**
### **Langkah 3.1 - Inisialisasi Model CNN**




In [14]:
cnn = tf.keras.models.Sequential()

### **Langkah 3.2 - Pembuatan Layer Konvulasi 1**



In [15]:
cnn.add(tf.keras.layers.Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=[7, 1]))

### **Langkah 3.3 - Pembuatan Layer Pooling 1**

In [16]:
cnn.add(tf.keras.layers.MaxPool1D(pool_size=2, strides=2))

### **Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2**

In [17]:
cnn.add(tf.keras.layers.Conv1D(filters=32, kernel_size=2, activation='relu'))
cnn.add(tf.keras.layers.MaxPool1D(pool_size=2, strides=2))

### **Langkah 3.5 - Flattening**

In [18]:
cnn.add(tf.keras.layers.Flatten())

### **Langkah 3.6 - Fully Connected Layer 1 (Input)**

In [19]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### **Langkah 3.7 - Fully Connected Layer 2 (Output)**

In [20]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### **Langkah 3.8 - Compile Model CNN**

In [21]:
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## **Langkah 4 - Fit CNN**

In [22]:
cnn.fit(x=X_train, y=y_train, validation_data=(X_test, y_test), epochs=25, batch_size=16)

Epoch 1/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6522 - loss: 0.6757 - val_accuracy: 0.6800 - val_loss: 0.6482
Epoch 2/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7324 - loss: 0.6315 - val_accuracy: 0.6800 - val_loss: 0.5982
Epoch 3/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7090 - loss: 0.5764 - val_accuracy: 0.8400 - val_loss: 0.5294
Epoch 4/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8294 - loss: 0.5130 - val_accuracy: 0.8800 - val_loss: 0.4582
Epoch 5/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8261 - loss: 0.4519 - val_accuracy: 0.8800 - val_loss: 0.3909
Epoch 6/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.4188 - val_accuracy: 0.8800 - val_loss: 0.3616
Epoch 7/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8662 - loss: 0.3817 - val_accuracy: 0.9200 - val_loss: 0.3158
Epoch 8/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8930 - loss: 0.3558 - val_accuracy: 0.9200 - val_los

## **Langkah 5 - Prediksi dengan 1 Citra**
Pada langkah ini, kita akan mencoba melakukan prediksi pada 1 citra anjing dan kucing.

In [23]:
# Contoh 1 data pasien baru: [Umur, Durasi Tidur, Kualitas Tidur, Aktivitas Fisik, Tingkat Stres, Detak Jantung, Langkah Harian]
test_data = np.array([[45, 5.8, 5, 30, 8, 82, 3500]])
test_data_scaled = scaler.transform(test_data)
test_data_reshaped = test_data_scaled.reshape((1, 7, 1))

result = cnn.predict(test_data_reshaped)

if result[0][0] >= 0.5:
    prediction = 'Gangguan Tidur (Insomnia / Sleep Apnea)'
else:
    prediction = 'Normal (Tidak Ada Gangguan Tidur)'

print(f"Probabilitas: {result[0][0]:.4f}")
print(f"Hasil Prediksi: {prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
Probabilitas: 0.9988
Hasil Prediksi: Gangguan Tidur (Insomnia / Sleep Apnea)


In [24]:
print(prediction)

Gangguan Tidur (Insomnia / Sleep Apnea)
